# Part I: Initial Benchmark
We are going to benchmark our current implementation, first we want to set $n$ and $m$ so that they can be on a reasonably sized instance. 

We use the ks_func function we implemented in HW2, and we set $n$  and $m$  to be 1,000,000 instead of 1000 to make the memory bottleneck and performance issues more observable.  And we use the benchmark tool to see the execute time and see what'll happen.



In [5]:
using Plots
using Revise
includet("C:/Users/Odysseus/GR6104_Homework/HW3/src/ks-stat.jl")

In [9]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000000)
m = randn(1000000)
@btime ks_func(n,m)
@benchmark ks_func(n,m)


  249.585 ms (116262 allocations: 97.33 MiB)


BenchmarkTools.Trial: 18 samples with 1 evaluation per sample.
 Range (min … max):  256.368 ms … 323.660 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     279.473 ms               ┊ GC (median):    7.65%
 Time  (mean ± σ):   285.287 ms ±  19.434 ms  ┊ GC (mean ± σ):  6.30% ± 5.42%

           ▃                            █                        
  ▇▁▁▁▇▁▁▇▁█▁▁▁▁▁▇▇▇▁▁▇▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▇▇▁▁▇▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▇ ▁
  256 ms           Histogram: frequency by time          324 ms <

 Memory estimate: 97.41 MiB, allocs estimate: 117655.

The current implementation takes approximately 170 ms to run. More importantly, it allocates 91.55 MiB of memory. This indicates a significant inefficiency, likely due to the creation of intermediate data structures during the labeling and sorting process.

# Part II: Iterative Code Optimization
## 1. Identify Bottlenecks

In [ ]:
ks_func(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n, m)

## Function and Line Number:
Based on the result from the Flame Graph, the primary bottleneck is located in the ks_func function, specifically at line 10 in ks-stat.jl. This line corresponds to the sort! operation on the combined array.

## Nature of the Bottleneck:
The bottleneck is primarily caused by unnecessary memory allocation, which leads to excessive computation:
    1. Memory Allocation caused by string, since the code creates a new tuple containing string (value,"label") for every single data point for a single run, it causes a huge volume of pressure on system's memory and the Collector.
    2. The Flame Graph shows that 74% of the execution time is spent in sort!. We might need to change the sort! a bit.

# 2. Alternative Implementations:

### Rationale for Alternatives
The bottleneck in the original code was the creation of intermediate `(value, label)` tuples and sorting a mixed array.
* **Alternative 1 (Separate Sorting + Two-Pointer):** Instead of mixing the arrays, we sort `X` and `Y` separately. This allows us to use Julia's highly optimized native float sorting. We then use a "Two-Pointer" algorithm to calculate the KS statistic in faster way which will avoid more allocations.

In [3]:
# Implementation for the alternatives two-pointer function 


function ks_func_2pt(X::AbstractVector,Y::AbstractVector)

    max_diff= 0
    cdf_x = 0
    cdf_y = 0
    n = length(X)
    m = length(Y)
    sort_x = sort(X)
    sort_y =sort(Y)
    i=1
    j=1

    while i<=n || j<=m 
        if i>n
            current_val = sort_y[j]
        elseif j>m
            current_val = sort_x[i]
        else 
            current_val = min(sort_x[i],sort_y[j])
        end
    

        while i <=n && sort_x[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sort_y[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff

end        

ks_func_2pt (generic function with 1 method)

# 3. Benchmark Alternatives:

In [10]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000000)
m = randn(1000000)
@btime ks_func_2pt(n,m)
@benchmark ks_func_2pt(n,m)

  101.445 ms (36642 allocations: 30.54 MiB)


BenchmarkTools.Trial: 34 samples with 1 evaluation per sample.
 Range (min … max):  123.824 ms … 205.479 ms  ┊ GC (min … max): 0.00% … 34.03%
 Time  (median):     142.234 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   147.424 ms ±  22.419 ms  ┊ GC (mean ± σ):  5.53% ± 10.24%

  ▃▃   ▃  ▃ ▃▃ ▃█▃ ▃                                             
  ██▁▇▇█▇▁█▇██▁███▁█▁▇▁▁▇▇▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▇▁▇▇▁▁▁▁▇ ▁
  124 ms           Histogram: frequency by time          205 ms <

 Memory estimate: 30.39 MiB, allocs estimate: 33639.

In [12]:
ks_func_2pt(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_2pt(n, m)

As we can see the alternative implementation significantly improved the bottleneck:

Execution Time: The median runtime dropped from 279.47 ms to 142.23 ms, achieving a ~2x speedup.

Memory Efficiency: Total memory allocation was reduced by 68% (from 97.33 MiB to 30.39 MiB).

Conclusion: The Flame Graph previously identified sort! as the main culprit (74% of time). By switching to a Two-Pointer approach with separate sorting of raw Float64 vectors, we bypassed the expensive process of sorting tuples with strings.

# 4. Update Code Base:

Therefore we have improved the bottleneck, we update our code base `ks_func` to `ks_func_2pt`. We re-run the unit tests to verify that nothing is broken.


In [14]:
# check for correctness

n = randn(1000)
m = randn(2000)

# run benchmarks
v1 = @btime ks_func(n,m)
v2 = @btime ks_func_2pt(n,m)

@assert isapprox(v1, v2)

  157.300 μs (13 allocations: 140.93 KiB)
  67.500 μs (23 allocations: 41.99 KiB)


We can see that the two results are approximately equal, we can say we are correct. And we update our `ks-stat,jl` in `src` and `ks-unit-test.jl` in `test`.

We can see that all the unit tests pass, nothing is broken , then our result is correct for the new optimization version. 

We commit and push our new code to GitHub.